# 02e · Circuits vs the full model on the whole dataset

**Question:** if a circuit has to answer **on its own**, how close is it to the full model, prompt by prompt?

Models compared on all 690 prompts (P, S, N; discovery and held-out):
| model | edges |
|---|---|
| full model | all 23,577 |
| EAP-IG candidate (Phase 1b) | 9600 |
| ACDC v1 (8 + 8 pairs, edge by edge) | 726 |
| ACDC v3b, τ = 0 | 7078 |
| ACDC v3b, τ = 0.005 | ≈ 5100 |
| baselines: no edges (the average prompt), random edges (size of v3b τ = 0) | |

**How a circuit answers alone: edge-level mean ablation.** The faithfulness of Phases 1b–2 needs a corrupted partner
for every prompt. Here the edges **not** in the circuit get the **dataset mean** instead:
receiver input = mean input + Σ over circuit edges (output(prompt) − mean output). Means are taken position by
position over the prompts with the same number of tokens (they share the chat template). All edges → exactly
the full model; no edges → the same answer for every prompt of a length.

**Caveat.** The circuits were found to explain the *difference* S vs P/N under patching from a corrupted
partner, not the absolute answer under mean ablation, so this is a harder, different test. The
threshold-free measures (triple accuracy, correlation with the full model) are the fairest.

**Measures** (per split; held-out = the new phrase "Great, just what I needed."):
* **Yes rate** per condition (d > 0): S should be Yes, P and N No;
* **accuracy** against the labels, and **triple accuracy**: d(S) > d(P) and d(S) > d(N);
* **agreement** with the full model's answer, and the **correlation** of d with the full model's d.

**Runtime:** any GPU with the 7B in 4-bit; about 10–15 minutes.

## Setup

In [ ]:
# Get (or update) the code from GitHub
import os, sys
REPO = "https://github.com/vasilis-pantzos/sarcasm-proposal.git"
ROOT = "/content/sarcasm-proposal"
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
sys.path.insert(0, ROOT)
%pip install -q -U accelerate bitsandbytes

In [ ]:
import os, json, time, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src.model import load_model, model_info, DEV_MODEL, MAIN_MODEL
from src.prompts import load_dataset, encode, chat_text
from src.eap import build_graph, random_mask
from src.standalone import MeanReference, StandaloneEvaluator, standalone_d
from src.metric import triple_accuracy
from src.utils import mount_drive, results_dir, cached, set_seed

# ---------------- settings ----------------
MODEL = MAIN_MODEL          # DEV_MODEL (3B) for quick tests
# ------------------------------------------
TAG = MODEL.split("/")[-1]
SUB = f"02e_compare_circuits/{TAG}"
mount_drive()
print("results go to:", results_dir(SUB))
set_seed(0)

In [ ]:
t0 = time.time()
model, tok = load_model(MODEL)
G = build_graph(model)
print(MODEL, model_info(model), f"loaded in {(time.time() - t0) / 60:.1f} min")

# the circuits (saved by notebooks 01b, 02 and 02d)
paths = {
    "EAP-IG candidate": f"01b_eap_ig/{TAG}/candidate_mask.npy",
    "ACDC v1": f"02_acdc/{TAG}/circuit_mask.npy",
    "ACDC v3b τ=0": f"02_acdc/{TAG}/v3b/acdc_mask_tau0.0.npy",
    "ACDC v3b τ=0.005": f"02_acdc/{TAG}/v3b/acdc_mask_tau0.005.npy",
}
CIRCUITS = {}
for name, p in paths.items():
    full = os.path.join(results_dir(""), p)
    if os.path.exists(full):
        CIRCUITS[name] = np.load(full)
    else:
        print("not found, skipped:", full)
size_ref = int(CIRCUITS.get("ACDC v3b τ=0", next(iter(CIRCUITS.values()))).sum())
CIRCUITS["random (same size as v3b τ=0)"] = random_mask(G, size_ref, np.random.default_rng(0))
CIRCUITS["no edges (average prompt)"] = np.zeros_like(G.valid)
for name, m in CIRCUITS.items():
    print(f"{name:34s} {int(m.sum()):6d} edges ({m.sum() / G.n_edges():.1%} of the graph)")

## Data and the mean reference

In [ ]:
df = load_dataset().sort_values(["split", "triple_id", "condition"]).reset_index(drop=True)
encs = [encode(tok, r) for r in df.itertuples()]
IDS = [e.ids for e in encs]
PREFIX = int(np.median([e.regions["situation"][0] for e in encs]))
SUFFIX = int(np.median([len(e.ids) - (e.regions["phrase"][-1] + 1) for e in encs]))
df["n_tok"] = [len(x) for x in IDS]
print(f"{len(df)} prompts; lengths {df.n_tok.min()}–{df.n_tok.max()}; template prefix {PREFIX}, suffix {SUFFIX} tokens")

t0 = time.time()
REF = MeanReference(model, IDS, G, prefix=PREFIX, suffix=SUFFIX)
print("mean reference per length (prompts):", REF.counts, f"| {time.time() - t0:.0f} s")

## Sanity check

All edges must give exactly the full model; no edges gives one answer per length.

In [ ]:
T0 = df.n_tok.mode()[0]
idx = list(df.index[df.n_tok == T0][:16])
ev = StandaloneEvaluator(model, tok, [IDS[i] for i in idx], REF, G)
gap_all = np.abs(ev.d(G.valid) - ev.d_clean).max()
d_none = ev.d(np.zeros_like(G.valid))
print(f"all edges: max |d - d_full| = {gap_all:.5f} | no edges: d from {d_none.min():+.3f} to {d_none.max():+.3f}")
assert gap_all < 1e-3, "sanity check failed: send the output to Claude"
del ev; torch.cuda.empty_cache()
print("PASS")

## Run every model on every prompt

In [ ]:
t0 = time.time()
key = "standalone_d_" + "_".join(f"{int(m.sum())}" for m in CIRCUITS.values())
D = cached(key, lambda: standalone_d(model, tok, IDS, CIRCUITS, REF, G), sub=SUB)
print(f"done in {(time.time() - t0) / 60:.1f} min")
MODELS = ["full", *CIRCUITS]
for k in MODELS:
    df[f"d {k}"] = D[k]

## Results

In [ ]:
def summary(sub):
    rows = []
    full_yes = sub["d full"] > 0
    for k in MODELS:
        d = sub[f"d {k}"]
        yes = d > 0
        piv = sub.pivot(index="triple_id", columns="condition", values=f"d {k}")
        rows.append({
            "model": k,
            "edges": G.n_edges() if k == "full" else int(CIRCUITS[k].sum()),
            "Yes rate P": yes[sub.condition == "P"].mean(),
            "Yes rate S": yes[sub.condition == "S"].mean(),
            "Yes rate N": yes[sub.condition == "N"].mean(),
            "accuracy": (yes == (sub.condition == "S")).mean(),
            "triple accuracy": triple_accuracy(piv.P, piv.S, piv.N),
            "agreement with full": (yes == full_yes).mean(),
            "corr. of d with full": np.corrcoef(d, sub["d full"])[0, 1] if d.std() > 0 else np.nan,
            "mean d (S) - mean d (P,N)": d[sub.condition == "S"].mean() - d[sub.condition != "S"].mean(),
        })
    return pd.DataFrame(rows).set_index("model")

TABLES = {split: summary(df[df.split == split]) for split in ["discovery", "heldout"]}
for split, t in TABLES.items():
    print(f"\n===== {split} ({(df.split == split).sum()} prompts) =====")
    display(t.round(3))

### Yes rate per condition (the ideal: P 0 %, S 100 %, N 0 %)

In [ ]:
cols = plt.cm.tab10(np.arange(len(MODELS)))
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (split, t) in zip(axes, TABLES.items()):
    w = 0.8 / len(MODELS)
    for i, k in enumerate(MODELS):
        ax.bar(np.arange(3) + (i - len(MODELS) / 2 + 0.5) * w, [t.loc[k, f"Yes rate {c}"] for c in "PSN"], w,
               color=cols[i], label=k)
    ax.set_xticks(range(3)); ax.set_xticklabels(["P (not sarcastic)", "S (sarcastic)", "N (not sarcastic)"])
    ax.set_title(f"{split} prompts"); ax.axhline(0.5, c="gray", lw=0.5, ls=":")
axes[0].set_ylabel("share of 'Yes'"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "yes_rate.png"), dpi=150); plt.show()

### Triple accuracy and agreement with the full model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, meas in zip(axes, ["triple accuracy", "agreement with full"]):
    x = np.arange(len(MODELS))
    for j, split in enumerate(TABLES):
        ax.bar(x + (j - 0.5) * 0.4, TABLES[split][meas].values, 0.4, label=split, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(MODELS, rotation=30, ha="right", fontsize=8)
    ax.set_title(meas); ax.set_ylim(0, 1.05)
axes[0].axhline(TABLES["discovery"].loc["full", "triple accuracy"], c="gray", ls="--", lw=0.8)
axes[0].legend()
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "triple_accuracy_agreement.png"), dpi=150); plt.show()

### d of each circuit vs d of the full model (every dot is a prompt)

In [ ]:
circ = [k for k in MODELS if k != "full"]
ccol = {"P": "#1565C0", "S": "#C62828", "N": "#2E7D32"}
fig, axes = plt.subplots(2, (len(circ) + 1) // 2, figsize=(4.2 * ((len(circ) + 1) // 2), 8))
for ax, k in zip(axes.flat, circ):
    for c in "PSN":
        s = df[df.condition == c]
        ax.scatter(s["d full"], s[f"d {k}"], s=6, alpha=0.5, color=ccol[c], label=c)
    lim = [df["d full"].min() - 1, df["d full"].max() + 1]
    ax.plot(lim, lim, c="gray", lw=0.6); ax.axhline(0, c="gray", lw=0.4); ax.axvline(0, c="gray", lw=0.4)
    r = TABLES["discovery"].loc[k, "corr. of d with full"]
    ax.set_title(f"{k}\n(r = {r:.2f} on discovery)", fontsize=9); ax.set_xlabel("d full model")
axes.flat[0].set_ylabel("d circuit"); axes.flat[0].legend(fontsize=7, markerscale=2)
for ax in list(axes.flat)[len(circ):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "d_vs_full.png"), dpi=150); plt.show()

### A few triples (held-out = the new phrase)

In [ ]:
rng = np.random.default_rng(1)
show = []
for split in ["discovery", "heldout"]:
    tids = rng.choice(df[df.split == split].triple_id.unique(), 3, replace=False)
    show.append(df[df.triple_id.isin(tids)])
cols_show = ["split", "triple_id", "condition", "text"] + [f"d {k}" for k in MODELS]
display(pd.concat(show)[cols_show].round(2))

## Try your own sentence

Write a sentence; every model answers (d > 0 = Yes). P(Yes) = probability of Yes among {Yes, No}.

In [ ]:
SENTENCE = "I spent the whole weekend fixing my car. What a fantastic way to relax."

template = df.prompt.iloc[0].replace(df.text.iloc[0], "{}")
ids = tok(chat_text(tok, template.format(SENTENCE)), add_special_tokens=False)["input_ids"]
ev = StandaloneEvaluator(model, tok, [ids], REF, G)
print(SENTENCE, f"\n({len(ids)} tokens" + ("" if ev.exact_mean else ", mean reference aligned from the closest length") + ")\n")
for k in MODELS:
    d = float(ev.d_clean[0]) if k == "full" else float(ev.d(CIRCUITS[k])[0])
    print(f"{k:34s} d = {d:+6.2f}  P(Yes) = {1 / (1 + np.exp(-d)):.2f}  -> {'Yes (sarcastic)' if d > 0 else 'No'}")
del ev; torch.cuda.empty_cache()

## Save

In [ ]:
d = results_dir(SUB)
df.drop(columns=["prompt"]).to_csv(os.path.join(d, "d_per_prompt.csv"), index=False)
for split, t in TABLES.items():
    t.to_csv(os.path.join(d, f"summary_{split}.csv"))
json.dump({split: t.round(4).reset_index().to_dict(orient="records") for split, t in TABLES.items()},
          open(os.path.join(d, "summary.json"), "w"), indent=1, ensure_ascii=False)
print("saved to", d)